In [11]:
import os
import re
import numpy as np
import pandas as pd
import glob

# -----------------------------
# CONFIG / INPUTS
# -----------------------------

#---------------- PROCESSING NETSURFP FILES -----------------------------
# 1. On définit UNIQUEMENT les colonnes dont on a besoin
COLS_TO_KEEP = ['id', ' n', ' rsa', ' q3']

netsurfp_files = glob.glob("netsurf*.csv")

# 2. On charge les fichiers en mode "économe"
df_list = []
for f in netsurfp_files:
    print(f"Chargement partiel de {f}...")
    # usecols permet de ne même pas lire les autres colonnes sur le disque
    tmp_df = pd.read_csv(f, usecols=COLS_TO_KEEP)
    df_list.append(tmp_df)

# Concaténation de petits morceaux
ns_all = pd.concat(df_list, ignore_index=True)

# 3. Nettoyage et Indexation (comme avant)
ns_all['clean_id'] = ns_all['id'].str.replace('>', '').str.split('_').str[0]
ns_all.set_index(['clean_id', ' n'], inplace=True)

print(f" Terminé ! NetSurfP contient {len(ns_all)} lignes indexées par (UniProt ID, Position).")


#------------recup uniprot ID et mutation sous format liste [ID, mutation, ID, mutation, etc]---------
df = pd.read_csv("Dataset_gbm.csv")
# 1 supprimer les lignes où l'ID ou la mutation serait manquant
df = df.dropna(subset=['UniProt ID', 'Mutation'])

# 2 Extraction et aplatissement en une seule ligne
liste_plate = df[['UniProt ID', 'Mutation']].values.flatten().tolist()
#--------------------------------------------------------------------------------------

INPUT_MUTATIONS = liste_plate
BASE_DIR = "data"
OUT_CSV = "final_sequence_dataset_features.csv"

MOTIFS = ['nM', 'Mc', 'n_M', 'M_c', 'n__M', 'M__c', 'tri']

AACON_HEADER = [
    'mut_pos', 'KABAT', 'JORES', 'SCHNEIDER', 'SHENKIN', 'GERSTEIN',
    'TAYLOR_GAPS', 'TAYLOR_NO_GAPS', 'VELIBIL', 'KARLIN', 'ARMON',
    'THOMPSON', 'NOT_LANCET', 'MIRNY', 'WILLIAMSON', 'LANDGRAF',
    'SANDER', 'VALDAR', 'SMERFS'
]

# -----------------------------
# LOAD REFERENCE TABLES
# -----------------------------
uniprot_seq = pd.read_excel(f"{BASE_DIR}\whole_proteome_uniprot_id.xlsx", engine="openpyxl")
uniprot_seq.rename({"From": "uniprot_ids"}, axis=1, inplace=True)
uniprot_seq["gene"] = uniprot_seq["Entry Name"].apply(lambda x: x.split("_")[0])


physchem_norm = pd.read_csv(fr"{BASE_DIR}\49_properties_normalizedValues.csv")
physchem_num = pd.read_csv(fr"{BASE_DIR}\49_properties_numerical_Values.csv")
extra_463 = pd.read_csv(fr"{BASE_DIR}\463_unique_numerical_properties.csv")

physchem_num_all = pd.concat([physchem_norm, extra_463[physchem_num.columns]], ignore_index=True)

mutation_matrices = pd.read_csv(fr"{BASE_DIR}\aaindex2_square_diagonal_properties.csv", keep_default_na=False)
mutation_matrices.set_index("wild_mut", inplace=True)

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def add_netsurfp_features(dict_prop, ns_all, uniprot_id, pos):
    """
    Associe les données structurales à la mutation en cours.
    """
    try:
        # On interroge l'index MultiIndex (ID, Position)
        row = ns_all.loc[(uniprot_id, pos)]
        
        # Ajout au dictionnaire de la mutation actuelle
        dict_prop[' rsa'] = float(row[' rsa'])
        
        # Encodage de la structure secondaire pour le ML
        # H=0 (Helix), E=1 (Strand), C=2 (Coil)
        q3_map = {'H': 0, 'E': 1, 'C': 2}
        dict_prop['secondary_structure'] = q3_map.get(row[' q3'], 2)
        
    except KeyError:
        # Si la protéine ou la position n'existe pas dans NetSurfP
        # on met des valeurs "neutres" pour ne pas fausser le modèle
        dict_prop[' rsa'] = 0.5  # Valeur moyenne par défaut
        dict_prop['secondary_structure'] = 2 # Coil par défaut

def get_sequence_and_gene(uniprot_id: str):
    seq = next(iter(uniprot_seq.loc[uniprot_seq["uniprot_ids"] == uniprot_id, "Sequence"]))
    gene = next(iter(uniprot_seq.loc[uniprot_seq["uniprot_ids"] == uniprot_id, "gene"]))
    return seq, gene

def window_13mer(sequence: str, pos_1based: int) -> str:
    pos0 = pos_1based - 1
    left = max(0, pos0 - 6)
    right = min(len(sequence), pos0 + 7)
    return sequence[left:right]

def safe_get(series, default=0):
    try:
        return next(iter(series))
    except StopIteration:
        return default

def compute_dipeptide_motifs(sequence: str, pos_1based: int, wild: str):
    pos0 = pos_1based - 1
    n1 = sequence[pos0 - 1] if pos0 - 1 >= 0 else '-'
    n2 = sequence[pos0 - 2] if pos0 - 2 >= 0 else '-'
    n4 = sequence[pos0 - 4] if pos0 - 4 >= 0 else '-'
    c1 = sequence[pos0 + 1] if pos0 + 1 < len(sequence) else '-'
    c2 = sequence[pos0 + 2] if pos0 + 2 < len(sequence) else '-'
    c0 = sequence[pos0 + 0] if pos0 < len(sequence) else '-'

    return {
        "nM": n1 + wild, 
        "Mc": wild + c0, 
        "tri": n1 + wild + c0,
        "n_M": n2 + "*" + wild, 
        "M_c": wild + "*" + c1,
        "n__M": (n4 + "**" + wild) if n4 != '-' else ('-' + wild),
        "M__c": wild + "**" + (c2 if c2 != '-' else c1)
    }

def add_physchem_features(dict_prop, wild, mut, win13):
    denom = sum(c.isalpha() for c in win13)
    a = sum(physchem_norm[win13[k]] for k in range(len(win13))) / denom
    b = sum(physchem_num_all[win13[k]] for k in range(len(win13))) / denom

    for idx, prop_name in enumerate(physchem_norm["index"]):
        w_norm = physchem_norm.loc[physchem_norm["index"] == prop_name, wild].values[0]
        m_norm = physchem_norm.loc[physchem_norm["index"] == prop_name, mut].values[0]
        dict_prop[f"{prop_name}_normalize_site_value"] = w_norm
        dict_prop[f"{prop_name}_normalize"] = a[idx]
        dict_prop[f"{prop_name}_normalize_diff"] = a[idx] - m_norm + w_norm

        w_num = physchem_num_all.loc[physchem_num_all["index"] == prop_name, wild].values[0]
        m_num = physchem_num_all.loc[physchem_num_all["index"] == prop_name, mut].values[0]
        dict_prop[f"{prop_name}_numeric_site_value"] = w_num
        dict_prop[f"{prop_name}_numeric_values"] = b[idx]
        dict_prop[f"{prop_name}_numeric_diff"] = b[idx] - m_num + w_num

    dict_prop.update({
        "neg_charge": len(re.findall("[DE]", win13)),
        "polar": len(re.findall("[NQSTP]", win13)),
        "aromatic": len(re.findall("[YFW]", win13)),
        "S_containing": len(re.findall("[CM]", win13)),
        "aliphatic": len(re.findall("[GALIV]", win13))
    })

def add_mutation_matrix_features(dict_prop, sequence, pos, wild, mut):
    pos0 = pos - 1
    prev_res = sequence[pos0 - 1] if pos0 - 1 >= 0 else "-"
    next_res = sequence[pos0 + 1] if pos0 + 1 < len(sequence) else "-"
    
    # Sécurité pour le résidu PRÉCÉDENT (N-term)
    try:
        prop_n = mutation_matrices.loc[mut + prev_res] - mutation_matrices.loc[wild + prev_res]
    except KeyError:
        # Si on est au début de la protéine, on met une différence de zéro
        prop_n = mutation_matrices.iloc[0] * 0 

    # Sécurité pour le résidu SUIVANT (C-term)
    try:
        prop_c = mutation_matrices.loc[mut + next_res] - mutation_matrices.loc[wild + next_res]
    except KeyError:
        # Si on est à la fin de la protéine, on met une différence de zéro
        prop_c = mutation_matrices.iloc[0] * 0

    for col in prop_n.index:
        dict_prop[col] = prop_n[col]
        dict_prop[col.lower()] = prop_c[col]

def add_odds_ratio_features(dict_prop, motifs):
    for motif in MOTIFS:
        x = pd.read_excel(f"{BASE_DIR}\GBM_odds_ratios_computed.xlsx", sheet_name=motif)
        row = x[x[motif] == motifs[motif]]
        odd = safe_get(row["odd_ratio"], default=0)
        
        conditions = [(row["odd_ratio"] >= 1.2) | (row["odd_ratio"] == "inf"),
                      (row["odd_ratio"] < 1.2) & (row["odd_ratio"] >= 0.9),
                      (row["odd_ratio"] < 0.9)]
        dict_prop[f"{motif}_coded_odds"] = safe_get(np.select(conditions, [1, 3, 2]), default=0)
        dict_prop[f"{motif}_odds_ratio"] = odd

def add_pssm_features(dict_prop, f4_lines, pos, wild, mut):
    cols = f4_lines[pos].strip().split()
    aa_order = list("ARNDCQEGHILKMFPSTWYV")
    score_map = {k: int(v) for k, v in zip(aa_order, cols[2:22])}

    dict_prop["pssm_score1"] = score_map.get(wild, 0)
    dict_prop["pssm_score2"] = sum(score_map.values()) / 20.0
    dict_prop["pssm_score3"] = score_map.get(mut, 0) - score_map.get(wild, 0)
    
    cons = f4_lines[pos].strip("\n")[90:].split()[21:22]
    dict_prop["conservation"] = float(cons[0]) if len(cons) else 0.0

# -----------------------------
# MAIN LOOP
# -----------------------------
all_rows = []
break_count = 0
print(f"Processing {len(INPUT_MUTATIONS)//2} mutations...")
for i in range(0, len(INPUT_MUTATIONS) - 1, 2):
    if i//2 % 1000 == 0:
        print("uniprot ID : ", INPUT_MUTATIONS[i], end=" | ")
        print("mutation : ", INPUT_MUTATIONS[i+1], end=" | ")
        print("mutation number : ", i//2 + 1, " / ", len(INPUT_MUTATIONS)//2)
    
    if INPUT_MUTATIONS[i] not in uniprot_seq["uniprot_ids"].values:
        break_count += 1
        print("on est dans le if", break_count)

    else:
        uniprot_id, mutation = INPUT_MUTATIONS[i], INPUT_MUTATIONS[i+1]
        wild, mut, pos = mutation[0], mutation[-1], int(mutation[1:-1])
        sequence, gene = get_sequence_and_gene(uniprot_id)

        if not (1 <= pos <= len(sequence)) or sequence[pos - 1] != wild:
            continue

        win13 = window_13mer(sequence, pos)
        motifs = compute_dipeptide_motifs(sequence, pos, wild)
        
        # Sequence-based auxiliary files
        #f4_lines = open(f"{BASE_DIR}\{uniprot_id}.pssm", "r").readlines()[2:]
        #f_aacon = pd.read_csv(f"{BASE_DIR}\{uniprot_id}.csv", skiprows=1, names=AACON_HEADER)

        dict_prop = {
            "UniProt ID": uniprot_id, "Gene Name": gene, "Mutation": mutation,
            "Wild": wild, "Mut": mut, "Pos": pos,
        }
        # 2. MATCHING NETSURFP (La nouvelle étape)
        add_netsurfp_features(dict_prop, ns_all, uniprot_id, pos)
        add_physchem_features(dict_prop, wild, mut, win13)
        add_mutation_matrix_features(dict_prop, sequence, pos, wild, mut)
        dict_prop.update(motifs)
        add_odds_ratio_features(dict_prop, motifs)
        #add_pssm_features(dict_prop, f4_lines, pos, wild, mut)

        """
        # Add AACon Features
        mm = f_aacon.iloc[pos - 1]
        for item in AACON_HEADER:
            dict_prop[item] = mm[item]
    """
        df_single_row = pd.DataFrame([dict_prop])
    
    # On ajoute au CSV. Le header n'est écrit que pour la toute première ligne réussie.
        df_single_row.to_csv(
            OUT_CSV, 
            mode='a', 
            index=False, 
            header=not os.path.exists(OUT_CSV) or os.stat(OUT_CSV).st_size == 0)

print(break_count, "mutations skipped due to missing UniProt ID.")
#pd.DataFrame(all_rows).to_csv(OUT_CSV, index=False)
print(f"Success! Sequence-only features saved to: {OUT_CSV}")

Chargement partiel de netsurfp_part1.csv...
Chargement partiel de netsurfp_part3.csv...
Chargement partiel de netsurfp_part4.csv...
Chargement partiel de netsurfp_part5.csv...
Chargement partiel de netsurf_part2.csv...
 Terminé ! NetSurfP contient 3598157 lignes indexées par (UniProt ID, Position).


c:\Users\nicol\miniconda3\envs\gioblastoma\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing 18114 mutations...
uniprot ID :  Q5JS54 | mutation :  L112P | mutation number :  1  /  18114
on est dans le if 1
on est dans le if 2
on est dans le if 3
on est dans le if 4
on est dans le if 5
on est dans le if 6
uniprot ID :  Q8IYG6 | mutation :  G102D | mutation number :  1001  /  18114
on est dans le if 7
on est dans le if 8
on est dans le if 9
on est dans le if 10
on est dans le if 11
on est dans le if 12
on est dans le if 13
on est dans le if 14
on est dans le if 15
on est dans le if 16
on est dans le if 17
on est dans le if 18
uniprot ID :  Q92900 | mutation :  A199T | mutation number :  2001  /  18114
on est dans le if 19
on est dans le if 20
uniprot ID :  O94832 | mutation :  T109M | mutation number :  3001  /  18114
on est dans le if 21
on est dans le if 22
on est dans le if 23
on est dans le if 24
on est dans le if 25
on est dans le if 26
on est dans le if 27
on est dans le if 28
on est dans le if 29
on est dans le if 30
on est dans le if 31
on est dans le if 32
on